# Experiment 2 — Full Context

## Configuration and imports

In [ ]:
from pathlib import Path
import ast, io, json, os, re, tokenize
import httpx
import pandas as pd
from IPython.display import display
from common import call_llm, load_gateway, parse_json_response, save_results_excel, score_sets, summarize_results, validate_l3_response

NOTEBOOK_DIR = Path.cwd()
WORKSPACE_DIR = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / "epic_gen.csv").exists() else NOTEBOOK_DIR
DATA_DIR = Path(os.getenv("L3_EXPERIMENT_DATA_DIR", WORKSPACE_DIR))
THEME_PATH = Path(os.getenv("L3_THEME_PATH", DATA_DIR / "epic_gen.csv"))
STAGE_PATH = Path(os.getenv("L3_STAGE_PATH", DATA_DIR / "VSSrv.csv"))
STAGE_CAPABILITY_MAP_PATH = Path(os.getenv("L3_STAGE_CAPABILITY_MAP_PATH", DATA_DIR / "VSSCaprv (1).csv"))
CAPABILITY_MASTER_PATH = Path(os.getenv("L3_CAPABILITY_MASTER_PATH", DATA_DIR / "VSSCaprv (1).csv"))
GROUND_TRUTH_PATH = Path(os.getenv("L3_GROUND_TRUTH_PATH", NOTEBOOK_DIR / "epic_l3_ground_truth_all_themes.xlsx"))
THEME_IDS = ["GROUP-15101", "GROUP-15097", "GROUP-15099"]
VALUE_STREAM_STAGE_FIELD_ID = "customfield_18700"
INSPECTION_THEME_ID = None
INSPECTION_EPIC_KEY = None
EXPERIMENT_NAME = "E2_FULL_CONTEXT"


## Theme and Epic retrieval

In [ ]:
def clean_text(v):
    if v is None or (not isinstance(v, (list, dict)) and pd.isna(v)): return ""
    return str(v).strip()

def parse_exported_list(v):
    if v is None or pd.isna(v): return []
    text = str(v).strip()
    if not text: return []
    if text.startswith("[") and text.endswith("]"):
        values = [ast.literal_eval(t.string) for t in tokenize.generate_tokens(io.StringIO(text).readline) if t.type == tokenize.STRING]
        if values: return [clean_text(x) for x in values]
    try: parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError): return [text]
    return [clean_text(x) for x in parsed] if isinstance(parsed, (list, tuple, set)) else [clean_text(parsed)]

def read_table(path):
    path = Path(path)
    if not path.exists(): raise FileNotFoundError(f"Required input not found: {path}")
    if path.suffix.lower() == ".csv": return pd.read_csv(path, dtype=str, encoding="cp1252", encoding_errors="replace")
    return pd.read_excel(path, dtype=str)

def load_themes(theme_ids=None):
    frame = read_table(THEME_PATH)
    if theme_ids: frame = frame.loc[frame["key"].isin(theme_ids)]
    themes = {}
    for _, row in frame.iterrows():
        keys = parse_exported_list(row.get("epic_keys")); desc = parse_exported_list(row.get("epic_description")); success = parse_exported_list(row.get("epic_successCriteria"))
        epics = [{"key": key, "description": desc[i] if i < len(desc) else "", "success_criteria": success[i] if i < len(success) else ""} for i, key in enumerate(keys)]
        themes[clean_text(row["key"])] = {"theme_description": clean_text(row.get("description")), "theme_business_needs": clean_text(row.get("businessNeeds")), "epics": epics}
    return themes

themes = load_themes(THEME_IDS)


## Value Stream Stage retrieval

In [ ]:
def jira_headers():
    serialized = os.getenv("JIRA_HEADERS_JSON")
    if serialized: return json.loads(serialized)
    token = os.getenv("JIRA_BEARER_TOKEN") or os.getenv("JIRA_TOKEN")
    if token: return {"Authorization": f"Bearer {token}", "Accept": "application/json"}
    raise RuntimeError("Set JIRA_HEADERS_JSON, JIRA_BEARER_TOKEN, or JIRA_TOKEN.")

def epic_stage_ids(epic_key):
    url = f"{os.environ['JIRA_BASE_URL'].rstrip('/')}/rest/api/2/issue/{epic_key}"
    verify = os.getenv("JIRA_VERIFY_SSL", "false").lower() == "true"
    with httpx.Client(headers=jira_headers(), verify=verify, timeout=30) as client:
        response = client.get(url, params={"fields": VALUE_STREAM_STAGE_FIELD_ID}); response.raise_for_status()
    raw = response.json().get("fields", {}).get(VALUE_STREAM_STAGE_FIELD_ID) or []
    raw = raw if isinstance(raw, list) else [raw]
    ids = []
    for value in raw:
        serialized = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else clean_text(value)
        ids.extend(re.findall(r"VSS\d+", serialized, flags=re.IGNORECASE))
    return list(dict.fromkeys(x.upper() for x in ids))

stage_frame = read_table(STAGE_PATH)
def stage_context(stage_id):
    row = stage_frame.loc[stage_frame["Value Stream Stage ID"].astype(str).str.strip() == stage_id]
    if row.empty: raise KeyError(f"No Value Stream Stage metadata found for {stage_id}.")
    row = row.iloc[0]
    return {"stage_id": stage_id, "stage_name": clean_text(row["Value Stream Stage Name"]), "stage_description": clean_text(row["Value Stream Stage Description"]), "entrance_criteria": clean_text(row["Value Stream Stage Entrance Criteria"]), "exit_criteria": clean_text(row["Value Stream Stage Exit Criteria"])}


## Stage → candidate L3 construction

In [ ]:
CANDIDATE_FIELDS = ["capability_id", "capability_name", "capability_description", "capability_tier"]
stage_capability_map = read_table(STAGE_CAPABILITY_MAP_PATH)
master = read_table(CAPABILITY_MASTER_PATH)[["Capability ID", "Capability Description", "Capability Tier"]].drop_duplicates("Capability ID")
def candidate_rows_for_stage(stage_id):
    rows = stage_capability_map.loc[stage_capability_map["Value Stream Stage ID"].astype(str).str.strip() == stage_id].copy()
    rows = rows.merge(master, on="Capability ID", how="left", sort=False, validate="many_to_one").sort_values(["Capability Name", "Capability ID"], kind="stable")
    return [{"capability_id": clean_text(r["Capability ID"]), "capability_name": clean_text(r["Capability Name"]), "capability_description": clean_text(r["Capability Description"]), "capability_tier": clean_text(r["Capability Tier"])} for _, r in rows.iterrows()]


## Production prompts

In [ ]:
SYSTEM_PROMPT = """You are an enterprise Business Capability Architecture specialist performing Level 3 (L3) business capability classification.

OBJECTIVE
Select only candidate L3 capabilities materially represented, enabled, changed, enhanced, or required by the supplied business context. This is capability classification, not keyword matching.

EVIDENCE PRIORITY
Use only fields that are present, in this order when available:
1. Epic success criteria
2. Epic description
3. Value Stream Stage context
4. Theme business needs
5. Theme description
If Epic context is absent, do not assume or refer to it.

CANDIDATE INTERPRETATION
- capability_description is the primary semantic definition.
- capability_name is the supporting label.
- capability_tier is supporting taxonomy context only.
- level_1_name and level_2_name, when supplied, are for disambiguation only and must never independently justify a selection.

DECISION PROCEDURE
1. Determine the core business function or outcome represented by the supplied context.
2. Compare it semantically against every candidate's capability_description.
3. Select a candidate only when direct evidence shows that its business function is materially represented; relatedness alone is insufficient.
4. Use Stage context to constrain/disambiguate, but Stage membership alone is not evidence.
5. Use Theme context as broader strategic context; it must not overpower more specific Epic evidence.
6. When candidates overlap, prefer the most specific directly aligned capability.

DO NOT SELECT merely because of shared keywords, hierarchy family, Stage membership, upstream/downstream relationship, data exchange, stakeholder involvement, technical adjacency, or general Theme relevance. Do not map technical implementation details unless the business capability itself is explicitly enabled or changed.

MULTI-SELECTION
Select 0 to 3 capabilities. Default to one when one capability adequately represents the function. Select multiple only for distinct material business functions with independent evidence. Return {"l3": []} when none is sufficiently supported.

REASONS
For every selection, give a concise reason connecting supplied evidence to the candidate definition. Use only exact capability_id values from the supplied candidates; never invent or alter an ID.

FINAL VALIDATION
Before responding, verify that every selected ID is a supplied candidate, every selection has direct evidence, no selection is merely adjacent, no stronger/more-specific candidate was omitted, multiple selections are genuinely distinct, and no more than three capabilities are selected.

OUTPUT CONTRACT
Return JSON only, with no Markdown, code fences, commentary, or extra fields:
{"l3":[{"capability_id":"CAP00000000","reason":"Concise evidence-based explanation."}]}"""


def build_user_prompt(theme, epic, stage, candidate_rows):
    payload = {
        "task": "Select the materially represented L3 business capabilities from the supplied candidates.",
        "theme": {"business_needs": theme["theme_business_needs"], "description": theme["theme_description"]},
        "epic": {"description": epic["description"], "success_criteria": epic["success_criteria"]},
        "value_stream_stage": stage,
        "candidate_l3_capabilities": candidate_rows,
        "selection_instruction": "Select 0 to 3 candidates; return an empty l3 list when none has direct evidence.",
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


## Prediction for one Stage

In [ ]:
def predict_for_stage(gateway, theme, epic, stage_id):
    stage = stage_context(stage_id); candidates = candidate_rows_for_stage(stage_id)
    user_prompt = build_user_prompt(theme, epic, stage, candidates)
    raw = call_llm(gateway, SYSTEM_PROMPT, user_prompt)
    selections = validate_l3_response(parse_json_response(raw), [c["capability_id"] for c in candidates], allow_empty=True, max_selected=3)
    return {"stage": stage, "candidates": candidates, "user_prompt": user_prompt, "raw_response": raw, "selections": selections}


## Single-example inspection

In [ ]:
if INSPECTION_THEME_ID and INSPECTION_EPIC_KEY:
    theme = themes[INSPECTION_THEME_ID]; epic = next(e for e in theme["epics"] if e["key"] == INSPECTION_EPIC_KEY)
    stage_id = epic_stage_ids(epic["key"])[0]; stage = stage_context(stage_id); candidates = candidate_rows_for_stage(stage_id)
    user_prompt = build_user_prompt(theme, epic, stage, candidates)
    print("SYSTEM PROMPT\n", SYSTEM_PROMPT); print("\nUSER PROMPT\n", user_prompt); display(pd.DataFrame(candidates))
    print("\nMODEL RESPONSE\n", predict_for_stage(load_gateway(), theme, epic, stage_id)["raw_response"])
else:
    print("Set INSPECTION_THEME_ID and INSPECTION_EPIC_KEY to run one sample.")


## Batch execution

Predictions are generated and aggregated at Epic level before ground truth is loaded.

In [ ]:
def run_predictions():
    gateway = load_gateway(); out = []
    for theme_id, theme in themes.items():
        for epic in theme["epics"]:
            stage_ids=[]; stage_predictions=[]; available=set(); predicted=set(); reasons=[]; status="ok"; error=None
            try:
                stage_ids = epic_stage_ids(epic["key"])
                for stage_id in stage_ids:
                    available.update(c["capability_id"] for c in candidate_rows_for_stage(stage_id))
                    result = predict_for_stage(gateway, theme, epic, stage_id)
                    stage_predictions.append({"stage_id": stage_id, "selections": result["selections"]})
                    for s in result["selections"]: predicted.add(s["capability_id"]); reasons.append({"stage_id": stage_id, **s})
            except Exception as exc: status="error"; error=str(exc)
            out.append({"experiment": EXPERIMENT_NAME, "theme_id": theme_id, "epic_key": epic["key"], "stage_ids": json.dumps(stage_ids), "available_candidate_l3_ids": json.dumps(sorted(available)), "predicted_l3_ids": json.dumps(sorted(predicted)), "model_reasons": json.dumps(reasons, ensure_ascii=False), "stage_predictions": json.dumps(stage_predictions, ensure_ascii=False), "status": status, "error": error})
    return pd.DataFrame(out)
predictions = run_predictions(); display(predictions.head(20))


## Ground-truth evaluation

Ground truth enters only in this post-prediction section.

In [ ]:
def ground_truth_by_epic():
    frame = read_table(GROUND_TRUTH_PATH)
    return {key: set(group["l3_capability_id"].dropna().astype(str).str.strip()) for key, group in frame.groupby("epic_key", sort=False)}
def evaluate_predictions(frame):
    truth = ground_truth_by_epic(); out=[]
    for row in frame.to_dict(orient="records"):
        pred=set(json.loads(row["predicted_l3_ids"])); available=set(json.loads(row["available_candidate_l3_ids"])); gt=truth.get(row["epic_key"])
        if row["status"]=="error": metrics={"exact_match":None,"precision":None,"recall":None,"f1":None,"predicted_count":len(pred),"truth_count":len(gt) if gt is not None else None}
        elif gt is None: row["status"]="missing_ground_truth"; metrics={"exact_match":None,"precision":None,"recall":None,"f1":None,"predicted_count":len(pred),"truth_count":None}
        else: metrics=score_sets(pred, gt)
        row["ground_truth_l3_ids"] = json.dumps(sorted(gt)) if gt is not None else None
        available_gt = gt & available if gt is not None else None
        row["gt_available_candidate_l3_ids"] = json.dumps(sorted(available_gt)) if available_gt is not None else None
        row["gt_candidate_available_count"] = len(available_gt) if available_gt is not None else None
        row["gt_candidate_availability_fraction"] = (len(available_gt)/len(gt) if gt else 1.0) if gt is not None else None
        row.update(metrics); out.append(row)
    return pd.DataFrame(out)
results = evaluate_predictions(predictions); display(summarize_results(results)); display(results.head(20)); print(f"Saved {save_results_excel(results, EXPERIMENT_NAME, 'results')}")
